# Function Calling — LLM jako "mózg" który wywołuje funkcje

## Jak deweloperzy budują AI asystentów i agenty?

### Plan warsztatu

1. **Idea**: LLM nie tylko generuje tekst — może *decydować* które funkcje wywołać
2. **Definiowanie narzędzi**: Opiszemy funkcje Pythona w formacie zrozumiałym dla LLM-a
3. **Pod maską**: Zobaczymy *dokładnie* co LLM widzi, jak wybiera funkcję, i co dostaje z powrotem
4. **Pętla agentowa**: Zbudujemy mini-agenta który sam decyduje jakie narzędzia użyć

### Czym jest Function Calling?

Normalnie LLM dostaje pytanie i generuje tekst. Ale co jeśli pytanie wymaga:
- Sprawdzenia **aktualnej** pogody?
- Wykonania **obliczeń** matematycznych?
- Przeszukania **bazy danych**?

LLM nie ma do tego dostępu — ale może **poprosić** nasz program o wywołanie odpowiedniej funkcji!

```
Użytkownik: "Jaka jest pogoda w Krakowie?"
     ↓
LLM myśli: "Muszę wywołać funkcję get_weather z argumentem city='Kraków'"
     ↓
Nasz kod: wywołuje get_weather('Kraków') → 15°C, słonecznie
     ↓
LLM: "W Krakowie jest 15°C i słonecznie."
```

Tak działają **ChatGPT Plugins**, **Claude Tools**, **GitHub Copilot** i wszelkie AI agenty!

## 1. Konfiguracja środowiska

In [1]:
# === Instalacja i import pakietów ===
import subprocess, sys

def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Instaluję {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("openai")

from openai import OpenAI
import requests
import json
import textwrap

print("Pakiety załadowane!")

Pakiety załadowane!


## 2. Połączenie z LLM-em

Tak samo jak w notebooku RAG — automatycznie szukamy lokalnej Ollamy lub serwera prowadzącego.

**Ważne:** Function calling działa najlepiej z modelami: `qwen2.5:7b+`, `mistral:7b+`, `llama3.1:8b+`

In [2]:
# === Auto-detekcja modelu LLM (identycznie jak w RAG) ===

PREFERRED_MODELS = [
    # Najlepsze modele do function calling
    "qwen3:14b", "qwen3:8b",
    "gemma4:12b", "gemma4:4b",
    "gemma3:27b", "gemma3:12b", "gemma3:8b",
    "qwen2.5:14b", "qwen2.5:7b",
    "gemma2:9b",
    "mistral:7b",
    "llama3.1:8b",
    # Małe modele — ostatnia deska ratunku dla słabszych laptopów
    "qwen3:4b", "qwen3:1.7b", "qwen3:0.6b",
    "gemma4:e4b", "gemma4:e2b",
    "gemma3:4b", "gemma3:1b",
    "qwen2.5:3b",
]

INSTRUCTOR_SERVER = "http://192.168.1.100:11434"  # <-- Prowadzący poda aktualny adres IP

def detect_ollama(base_url="http://localhost:11434"):
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=3)
        if r.status_code == 200:
            return [m["name"] for m in r.json().get("models", [])]
    except:
        pass
    return []

def pick_best_model(available_models):
    for preferred in PREFERRED_MODELS:
        for available in available_models:
            if preferred.split(":")[0] in available and (":" not in preferred or preferred in available):
                return available
    return available_models[0] if available_models else None

print("Szukam lokalnej Ollamy...")
local_models = detect_ollama("http://localhost:11434")

if local_models:
    OLLAMA_URL = "http://localhost:11434"
    MODEL_NAME = pick_best_model(local_models)
    print(f"Lokalna Ollama znaleziona! Model: {MODEL_NAME}")
else:
    print("Lokalna Ollama niedostępna. Próbuję serwer prowadzącego...")
    instructor_models = detect_ollama(INSTRUCTOR_SERVER)
    if instructor_models:
        OLLAMA_URL = INSTRUCTOR_SERVER
        MODEL_NAME = pick_best_model(instructor_models)
        print(f"Połączono z serwerem! Model: {MODEL_NAME}")
    else:
        print("UWAGA: Brak dostępnego LLM-a! Zainstaluj Ollama lub poproś o adres serwera.")
        OLLAMA_URL = None
        MODEL_NAME = None

if OLLAMA_URL:
    client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")
    print(f"\nKlient LLM gotowy!")

Szukam lokalnej Ollamy...
Lokalna Ollama znaleziona! Model: mistral:7b

Klient LLM gotowy!


## 3. Definiujemy nasze "narzędzia" (funkcje Pythona)

Zaczniemy od prostych funkcji, które LLM będzie mógł wywoływać.
Każda funkcja ma **docstring** — opis co robi. To jest kluczowe!
LLM czyta ten opis żeby zdecydować *kiedy* i *jak* użyć danej funkcji.

Zobaczmy co się stanie **pod maską**.

In [ ]:
# === Nasze narzędzia (zwykłe funkcje Pythona) ===

import random
import math

def get_weather(city: str) -> str:
    """
    Sprawdza aktualną pogodę w podanym mieście.
    Zwraca temperaturę i opis pogody.
    
    Args:
        city: Nazwa miasta, np. 'Kraków', 'Warszawa', 'Gdańsk'
    """
    # Symulacja — w prawdziwym systemie tu byłoby API pogodowe
    pogoda = {
        "Kraków": {"temp": 18, "opis": "słonecznie", "wilgotność": 45},
        "Warszawa": {"temp": 15, "opis": "pochmurno", "wilgotność": 70},
        "Gdańsk": {"temp": 12, "opis": "deszcz", "wilgotność": 85},
        "Wrocław": {"temp": 20, "opis": "słonecznie", "wilgotność": 40},
        "Poznań": {"temp": 16, "opis": "częściowe zachmurzenie", "wilgotność": 55},
    }
    if city in pogoda:
        p = pogoda[city]
        return f"Pogoda w {city}: {p['temp']}°C, {p['opis']}, wilgotność {p['wilgotność']}%"
    return f"Nie mam danych pogodowych dla miasta: {city}"


def calculate(expression: str) -> str:
    """
    Wykonuje obliczenie matematyczne.
    Przyjmuje wyrażenie jako tekst i zwraca wynik.
    
    Args:
        expression: Wyrażenie matematyczne, np. '2 + 2', 'sqrt(144)', '15 * 7'
    """
    try:
        # Bezpieczna ewaluacja (tylko operacje matematyczne)
        allowed = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos, 
                   "pi": math.pi, "abs": abs, "round": round, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"Wynik: {expression} = {result}"
    except Exception as e:
        return f"Błąd w obliczeniu '{expression}': {e}"


def search_presidents(query: str) -> str:
    """
    Przeszukuje bazę danych o prezydentach Polski.
    Znajduje informacje pasujące do zapytania.
    
    Args:
        query: Zapytanie, np. 'najmłodszy prezydent', 'katastrofa lotnicza', 'Kwaśniewski'
    """
    # Prosta baza danych (w produkcji tu byłby np. RAG z poprzedniego notebooka!)
    baza = [
        {"prezydent": "Lech Wałęsa", "kadencja": "1990-1995", "info": "Pierwszy prezydent wybrany w wolnych wyborach. Laureat Pokojowej Nagrody Nobla."},
        {"prezydent": "Aleksander Kwaśniewski", "kadencja": "1995-2005", "info": "Najmłodszy prezydent III RP (41 lat). Sprawował urząd przez dwie kadencje (3653 dni)."},
        {"prezydent": "Lech Kaczyński", "kadencja": "2005-2010", "info": "Zginął w katastrofie lotniczej pod Smoleńskiem 10 kwietnia 2010 r."},
        {"prezydent": "Bronisław Komorowski", "kadencja": "2010-2015", "info": "Najstarszy prezydent III RP w dniu zaprzysiężenia (58 lat)."},
        {"prezydent": "Andrzej Duda", "kadencja": "2015-2025", "info": "Sprawował urząd przez dwie kadencje."},
        {"prezydent": "Rafał Trzaskowski", "kadencja": "od 2025", "info": "Aktualny prezydent RP."},
    ]
    
    query_lower = query.lower()
    wyniki = [p for p in baza if query_lower in json.dumps(p, ensure_ascii=False).lower()]
    
    if wyniki:
        return "\n".join([f"• {p['prezydent']} ({p['kadencja']}): {p['info']}" for p in wyniki])
    return f"Nie znaleziono wyników dla zapytania: {query}"


# Słownik naszych narzędzi — mapowanie nazwy na funkcję
AVAILABLE_TOOLS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_presidents": search_presidents,
}

print("Zdefiniowano 3 narzędzia:")
for name, func in AVAILABLE_TOOLS.items():
    # Pokazujemy docstring — to jest TO CO LLM "WIDZI" o każdej funkcji!
    print(f"\n  {name}():")
    print(f"    {func.__doc__.strip().split(chr(10))[0]}")

## 4. Opisanie narzędzi dla LLM-a (schemat JSON)

LLM nie widzi naszego kodu Pythona bezpośrednio. Musimy mu **opisać** 
każde narzędzie w formacie JSON Schema — standardzie którego używają:
- OpenAI (ChatGPT)
- Anthropic (Claude)
- Ollama (lokalne modele)

Zobaczmy jak wygląda taki opis — to jest **dokładnie** to co LLM dostaje:

In [ ]:
# === Definicje narzędzi w formacie OpenAI/Ollama ===
# To jest opis, który LLM "czyta" żeby wiedzieć jakie ma narzędzia

tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Sprawdza aktualną pogodę w podanym mieście. Użyj gdy użytkownik pyta o pogodę.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nazwa miasta, np. 'Kraków', 'Warszawa'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policzyć.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Wyrażenie matematyczne, np. '2+2', 'sqrt(144)'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_presidents",
            "description": "Przeszukuje bazę danych o prezydentach Polski. Użyj gdy pytanie dotyczy prezydentów RP.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Zapytanie do bazy, np. 'Kwaśniewski', 'katastrofa'"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("=== TO WIDZI LLM (definicje narzędzi) ===")
print(json.dumps(tools_definition, indent=2, ensure_ascii=False))

## 5. Pierwszy function call — krok po kroku, pod maską!

Teraz wyślemy pytanie do LLM-a razem z opisem narzędzi.
LLM **sam zdecyduje** czy potrzebuje narzędzia i którego.

Zobaczmy **dokładnie** co się dzieje na każdym etapie — 
każdą wiadomość, każdą decyzję LLM-a.

In [ ]:
# === Krok po kroku: Function Calling pod maską ===

user_question = "Jaka jest pogoda w Krakowie?"

if OLLAMA_URL:
    print("╔" + "═"*68 + "╗")
    print("║  KROK 1: Wysyłamy pytanie + opisy narzędzi do LLM-a            ║")
    print("╚" + "═"*68 + "╝")
    print(f"\n  Pytanie użytkownika: \"{user_question}\"")
    print(f"  Dostępne narzędzia: {[t['function']['name'] for t in tools_definition]}")
    print(f"  → Wysyłam do {MODEL_NAME}...\n")
    
    # Wysyłamy pytanie Z definicjami narzędzi
    messages = [
        {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku. Używaj narzędzi gdy to potrzebne."},
        {"role": "user", "content": user_question}
    ]
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools_definition,
        temperature=0.1,
    )
    
    assistant_message = response.choices[0].message
    
    print("╔" + "═"*68 + "╗")
    print("║  KROK 2: LLM odpowiada — czy chce wywołać funkcję?             ║")
    print("╚" + "═"*68 + "╝")
    
    if assistant_message.tool_calls:
        tool_call = assistant_message.tool_calls[0]
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        print(f"\n  TAK! LLM chce wywołać funkcję!")
        print(f"  Funkcja: {func_name}")
        print(f"  Argumenty: {func_args}")
        print(f"  (LLM sam wybrał funkcję i argumenty na podstawie opisu!)\n")
        
        print("╔" + "═"*68 + "╗")
        print("║  KROK 3: Nasz kod wywołuje funkcję i zwraca wynik            ║")
        print("╚" + "═"*68 + "╝")
        
        # Wywołujemy naszą prawdziwą funkcję Pythona!
        result = AVAILABLE_TOOLS[func_name](**func_args)
        print(f"\n  Wywołanie: {func_name}({func_args})")
        print(f"  Wynik: \"{result}\"\n")
        
        print("╔" + "═"*68 + "╗")
        print("║  KROK 4: Wynik wraca do LLM-a → generuje odpowiedź           ║")
        print("╚" + "═"*68 + "╝")
        
        # Dodajemy wynik funkcji do konwersacji i wysyłamy ponownie
        messages.append(assistant_message)  # odpowiedź LLM-a z tool_call
        messages.append({                    # wynik naszej funkcji
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
        
        # LLM generuje finalną odpowiedź na podstawie wyniku
        final_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.1,
        )
        
        final_answer = final_response.choices[0].message.content
        print(f"\n  Ostateczna odpowiedź LLM-a:")
        print(f"  \"{final_answer}\"")
    else:
        print(f"\n  LLM odpowiedział bez użycia narzędzi:")
        print(f"  \"{assistant_message.content}\"")
    
    print("\n" + "═"*70)
    print("Cała konwersacja (to co widziała sieć neuronowa):")
    print("═"*70)
    for msg in messages:
        role = msg["role"] if isinstance(msg, dict) else msg.role
        if isinstance(msg, dict):
            content = msg.get("content", "[tool_calls]")
        else:
            content = msg.content or f"[wywołanie: {msg.tool_calls[0].function.name}({msg.tool_calls[0].function.arguments})]"
        print(f"  [{role:>10}]: {str(content)[:100]}")
else:
    print("LLM niedostępny.")

## 6. LLM sam wybiera narzędzie!

Zobaczmy jak LLM reaguje na **różne** pytania — 
automatycznie wybiera odpowiednie narzędzie (lub żadne!).

In [ ]:
# === Automatyczne function calling dla różnych pytań ===

def ask_with_tools(question, verbose=True):
    """
    Wysyła pytanie do LLM-a z narzędziami.
    Automatycznie obsługuje function calling.
    Jeśli verbose=True, pokazuje każdy krok.
    """
    if not OLLAMA_URL:
        print("LLM niedostępny.")
        return None
    
    messages = [
        {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku. Używaj narzędzi gdy to potrzebne."},
        {"role": "user", "content": question}
    ]
    
    response = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.1
    )
    
    assistant_msg = response.choices[0].message
    
    # Jeśli LLM nie chce narzędzia — zwróć odpowiedź
    if not assistant_msg.tool_calls:
        if verbose:
            print(f"  Narzędzie: BRAK (LLM odpowiedział sam)")
            print(f"  Odpowiedź: {assistant_msg.content[:200]}")
        return assistant_msg.content
    
    # Obsłuż function call
    tool_call = assistant_msg.tool_calls[0]
    func_name = tool_call.function.name
    func_args = json.loads(tool_call.function.arguments)
    
    if verbose:
        print(f"  Narzędzie: {func_name}({func_args})")
    
    # Wywołaj funkcję
    result = AVAILABLE_TOOLS[func_name](**func_args)
    if verbose:
        print(f"  Wynik:     {result}")
    
    # Wyślij wynik z powrotem do LLM-a
    messages.append(assistant_msg)
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    
    final = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.1)
    final_answer = final.choices[0].message.content
    
    if verbose:
        print(f"  Odpowiedź: {final_answer[:200]}")
    
    return final_answer

print("Funkcja ask_with_tools() gotowa!")

In [ ]:
# === Testujemy różne pytania ===

pytania = [
    "Jaka jest pogoda w Gdańsku?",                          # → get_weather
    "Ile to jest 17 razy 23?",                               # → calculate
    "Kto był najmłodszym prezydentem Polski?",               # → search_presidents
    "Co to jest Python?",                                     # → żadne narzędzie!
    "Oblicz pierwiastek z 256 i powiedz mi pogodę w Poznaniu",  # → dwa narzędzia?
]

if OLLAMA_URL:
    for q in pytania:
        print(f"\n{'═'*60}")
        print(f"PYTANIE: {q}")
        print(f"{'═'*60}")
        ask_with_tools(q)
else:
    print("LLM niedostępny.")

### Ćwiczenie 1: Dodaj własne narzędzie

Stwórz nową funkcję i dodaj jej opis do `tools_definition`.

Pomysły:
- `translate(text, target_language)` — "tłumacz" (może użyć słownika)
- `get_population(city)` — zwraca liczbę mieszkańców
- `random_joke()` — zwraca losowy dowcip

**Ważne:** Dobrze napisany `description` w definicji narzędzia jest kluczowy — 
to on decyduje czy LLM wybierze Twoje narzędzie!

In [ ]:
# === Ćwiczenie 1: Dodaj własne narzędzie ===

### Proszę poniżej uzupełnić kod ### (≈ 15 linijek)
# Krok 1: Zdefiniuj funkcję Pythona

def get_population(city: str) -> str:
    """
    Zwraca przybliżoną liczbę mieszkańców miasta w Polsce.
    
    Args:
        city: Nazwa miasta, np. 'Kraków', 'Warszawa'
    """
    populacja = {
        "Warszawa": 1_860_000,
        "Kraków": 800_000,
        "Wrocław": 670_000,
        "Gdańsk": 470_000,
        "Poznań": 535_000,
    }
    if city in populacja:
        return f"{city} ma około {populacja[city]:,} mieszkańców."
    return f"Nie mam danych o populacji dla: {city}"

# Krok 2: Dodaj do dostępnych narzędzi
AVAILABLE_TOOLS["get_population"] = get_population

# Krok 3: Dodaj opis w formacie JSON Schema
tools_definition.append({
    "type": "function",
    "function": {
        "name": "get_population",
        "description": "Zwraca liczbę mieszkańców miasta w Polsce. Użyj gdy pytanie dotyczy populacji.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nazwa miasta"}
            },
            "required": ["city"]
        }
    }
})

### Koniec uzupełniania kodu ###

print(f"Teraz mamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")

# Przetestuj!
if OLLAMA_URL:
    print("\nTest:")
    ask_with_tools("Ile mieszkańców ma Kraków?")

## 7. Pętla agentowa — LLM który myśli i działa

Do tej pory obsługiwaliśmy **jedno** wywołanie narzędzia.
Ale co jeśli LLM potrzebuje **kilku** narzędzi po kolei?

Np. pytanie: *"Jaka jest pogoda w mieście z największą populacją?"*
wymaga:
1. Sprawdzenia populacji miast
2. Wybrania największego
3. Sprawdzenia pogody w tym mieście

To jest **pętla agentowa** — LLM wielokrotnie wywołuje narzędzia aż znajdzie odpowiedź.

```
while LLM chce wywołać narzędzie:
    1. LLM wybiera narzędzie i argumenty
    2. Nasz kod wywołuje funkcję
    3. Wynik wraca do LLM-a
    4. LLM decyduje: potrzebuję jeszcze jednego narzędzia? → wróć do 1.
                     mam wszystko? → generuję odpowiedź
```

In [ ]:
# === Pętla agentowa z pełnym logowaniem ===

def agent(question, max_steps=5):
    """
    Mini-agent: LLM z narzędziami w pętli.
    Pokazuje każdy krok "pod maską".
    """
    if not OLLAMA_URL:
        print("LLM niedostępny.")
        return
    
    messages = [
        {"role": "system", "content": "Jesteś pomocnym asystentem. Odpowiadaj po polsku. "
         "Używaj narzędzi aby znaleźć potrzebne informacje. "
         "Możesz wywołać wiele narzędzi po kolei."},
        {"role": "user", "content": question}
    ]
    
    print(f"Pytanie: {question}")
    print(f"{'─'*60}")
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.1
        )
        
        msg = response.choices[0].message
        
        if not msg.tool_calls:
            # LLM zakończył — ma odpowiedź
            print(f"\n  Krok {step+1}: LLM generuje ostateczną odpowiedź")
            print(f"  {'─'*50}")
            print(f"  {msg.content}")
            return msg.content
        
        # LLM chce wywołać narzędzie
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"\n  Krok {step+1}: LLM wywołuje → {func_name}({func_args})")
            
            # Wywołaj funkcję
            result = AVAILABLE_TOOLS[func_name](**func_args)
            print(f"           Wynik ← \"{result}\"")
            
            # Dodaj do konwersacji
            messages.append(msg)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    
    print(f"\n  (Osiągnięto limit {max_steps} kroków)")

print("Agent gotowy! Użyj: agent('twoje pytanie')")

In [ ]:
# === Testujemy agenta! ===

if OLLAMA_URL:
    print("Test 1: Proste pytanie (jedno narzędzie)")
    print("═"*60)
    agent("Kto zginął w katastrofie lotniczej i w jakim mieście jest teraz najcieplej?")
    
    print("\n\n")
    print("Test 2: Pytanie wymagające obliczeń")
    print("═"*60)
    agent("Ile lat miał Aleksander Kwaśniewski gdy skończył kadencję? Oblicz ile to dni.")

### Ćwiczenie 2: Zadaj agentowi trudne pytanie

Wymyśl pytanie, które wymaga użycia **dwóch lub więcej** narzędzi.
Obserwuj jak agent krok po kroku dochodzi do odpowiedzi.

In [ ]:
# === Ćwiczenie 2: Twoje pytanie do agenta ===

### Proszę poniżej uzupełnić kod ### (≈ 1 linijka)
# Wskazówka: wymyśl pytanie łączące pogodę, obliczenia, i prezydentów

MOJE_PYTANIE = "Ile lat temu zginął Lech Kaczyński i jaka jest dziś pogoda w Krakowie?"  # <-- ZMIEŃ!

### Koniec uzupełniania kodu ###

if OLLAMA_URL:
    agent(MOJE_PYTANIE)

## 8. Podsumowanie

### Co zrobiliśmy?

1. **Zdefiniowaliśmy narzędzia** — zwykłe funkcje Pythona z opisami
2. **Opisaliśmy je dla LLM-a** — w formacie JSON Schema (standard branżowy)
3. **Zobaczyliśmy pod maską** — co dokładnie LLM widzi, jak wybiera funkcję, jakie argumenty generuje
4. **Zbudowaliśmy pętlę agentową** — LLM sam decyduje które narzędzia wywołać i w jakiej kolejności

### Tak działają prawdziwe produkty AI

| Produkt | Narzędzia |
|---|---|
| ChatGPT | Przeglądarka, DALL-E, Code Interpreter, Plugins |
| Claude | Wyszukiwanie, analiza plików, kod, MCP |
| GitHub Copilot | Czytanie/pisanie plików, terminal, wyszukiwanie kodu |
| Cursor / Windsurf | Edycja kodu, terminal, dokumentacja |

Wszystkie działają na tej samej zasadzie: **LLM + zestaw narzędzi + pętla agentowa**.

### Co dalej?

Połączenie tego co widzieliśmy:
- **RAG** (z poprzedniego notebooka) może być jednym z *narzędzi* agenta!
- Agent z narzędziem `search_documents()` = inteligentny asystent dokumentacji
- Dodaj narzędzie `send_email()` = agent do obsługi klienta
- Dodaj narzędzie `write_code()` = asystent programistyczny

To jest sposób w jaki deweloperzy budują nowoczesne aplikacje AI.